In [2]:
import polars as pl

ruta = "/Users/macbook/ProyectosLocales/PrecioLuz/datos/demanda_energia.csv"

df = pl.read_csv(ruta, separator=";")

# columnas necesarias
df = df.select(["datetime", "value"])

# fecha
df = df.with_columns(
    pl.col("datetime")
    .str.slice(0, 19)
    .str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S")
    .dt.date()
    .alias("date")
).drop("datetime")

df = df.with_columns(
    pl.col("value").cast(pl.Float64)
).sort("date")

# limpiar
nulos = df.null_count()
dup = df.height - df.unique(subset=["date"]).height

df = df.drop_nulls().unique(subset=["date"])

print("Filas:", df.height)
print("Rango:", df["date"].min(), df["date"].max())

print(df.select([
    pl.col("value").min().alias("min"),
    pl.col("value").max().alias("max"),
    pl.col("value").mean().alias("media")
]))

print("Nulos:", nulos)
print("Duplicados:", dup)

# faltantes
fechas = pl.date_range(
    pl.date(2015,1,1),
    pl.date(2025,12,31),
    "1d",
    eager=True
)

faltan = pl.DataFrame({"date": fechas}).join(df, on="date", how="anti")

print("Faltan:", faltan.height)

# guardar
df.write_csv("/Users/macbook/ProyectosLocales/PrecioLuz/datos/demanda_total.csv")

Filas: 4018
Rango: 2015-01-01 2025-12-31
shape: (1, 3)
┌──────────────┬──────────────┬──────────────┐
│ min          ┆ max          ┆ media        │
│ ---          ┆ ---          ┆ ---          │
│ f64          ┆ f64          ┆ f64          │
╞══════════════╪══════════════╪══════════════╡
│ 15512.229167 ┆ 35306.409722 ┆ 27725.371516 │
└──────────────┴──────────────┴──────────────┘
Nulos: shape: (1, 2)
┌───────┬──────┐
│ value ┆ date │
│ ---   ┆ ---  │
│ u32   ┆ u32  │
╞═══════╪══════╡
│ 0     ┆ 0    │
└───────┴──────┘
Duplicados: 0
Faltan: 0
